# 练习实验 - 探索大语言模型 (LLM) 的能力

---

欢迎来到探索大语言模型 (LLM) 参数能力的练习实验！在本实验中，你将研究不同的参数如何影响 LLM 的输出，从而使你能够生成更多样化的结果。你还将学习如何开发一种方法，让 LLM 能够维持对话上下文，像聊天机器人一样运行！

1. 开发一个能够让 LLM 维持连贯对话上下文的函数。
2. 探索不同参数如何影响 LLM 的行为和输出。

---
<h4 style="color:black; font-weight:bold;">使用目录</h4>

JupyterLab 为你提供了一种在作业中导航的简便方式。它位于左侧面板的“目录” (Table of Contents) 选项卡下，如下图所示。

![目录位置](images/toc.png)

---

# 目录
- [ 1 - 导入库](#1)
- [ 2 - 生成函数回顾](#2)
  - [ 2.1 `generate_with_single_input` 和 `generate_with_multiple_input`](#2-1)
  - [ 2.2 使用所需参数生成 kwargs](#2-2)
  - [ 2.3 允许 LLM 保持对话 ](#2-3)
- [ 3 - 理解参数](#3)
  - [ 3.1 简介](#3-1)
  - [ 3.2 核采样 - `top_p`](#3-2)
  - [ 3.3 Top-k 采样](#3-3)
  - [ 3.4 温度](#3-4)
  - [ 3.5 重复惩罚](#3-5)
- [ 4 - 额外：创建一个简单的聊天机器人](#4)

<a id='1'></a>
## 1 - 导入库

运行下面的单元格以导入必要的库。

In [1]:
import json
import random

In [2]:
# 从本地或库文件 utils.py 中导入两个关键的生成函数
from utils import (
    # generate_with_single_input: 用于处理单次输入（Prompt）并返回结果
    # 适用于标准问答场景，将检索到的上下文和问题一次性喂给 AI
    generate_with_single_input, 
    
    # generate_with_multiple_input: 用于处理多轮对话或复杂的输入列表（Messages）
    # 适用于需要维持上下文历史（History）的交互式对话场景
    generate_with_multiple_input
)

<a id='2'></a>
## 2 - 生成函数回顾


<a id='2-1'></a>
### 2.1 `generate_with_single_input` 和 `generate_with_multiple_input`

让我们回顾一下你在本课程中一直使用的生成函数。

```Python
generate_with_single_input(prompt: str, 
                               role: str = 'user', 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")


generate_with_multiple_input(messages: List[Dict], 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")

In [3]:
# 调用生成函数，将关于“分圆多项式（Cyclotomic Polynomial）”的提问发送给模型
# generate_with_single_input 会处理 API 通信，并返回一个包含 AI 回答的字典对象
# 约束条件：非常简短（very briefly），且不超过 5 句话
generate_with_single_input("Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences.")

# 作用：
# 1. 验证生成接口：确保 LLM 能够理解并遵守你设定的“简短”和“句数限制”等约束。
# 2. 零样本测试 (Zero-shot)：这属于不提供 Context 的直接提问，用于观察模型的原始知识储备。

{'role': 'assistant',
 'content': 'A cyclotomic polynomial \\(\\Phi_n(x)\\) is the minimal monic polynomial over the rationals whose roots are the primitive \\(n\\)th roots of unity—i.e., complex numbers \\(z\\) such that \\(z^n = 1\\) but \\(z^k \\neq 1\\) for any \\(0 < k < n\\). It has integer coefficients and degree \\(\\phi(n)\\), where \\(\\phi\\) is Euler’s totient function. Cyclotomic polynomials are irreducible over \\(\\mathbb{Q}\\) and satisfy the factorization \\(x^n - 1 = \\prod_{d \\mid n} \\Phi_d(x)\\). They play a central role in algebraic number theory, Galois theory, and constructions of regular polygons.'}

`generate_with_multiple_input` 函数接收格式为 `{'role': role, 'content': prompt}` 的消息列表。此函数允许你**创建上下文**。

In [4]:
# 定义系统角色（System Role）：这是 AI 的“性格底色”
# 这里将 AI 设定为一个“极具讽刺意味但乐于助人”的助手
system_dict = {
    "role": 'system', 
    'content': 'You are a very ironic, but helpful assistant.'
}

# 定义用户角色（User Role）：即用户提出的具体问题
user_dict = {
    "role": "user", 
    'content': "Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences."
}

# 将消息按顺序封装进一个列表，构成对话历史（Conversation History）
messages = [system_dict, user_dict]

# 调用多消息生成函数
# 这种方式不仅让模型知道你问了什么，还让它知道它应该“以什么样的姿态”来回答你
generate_with_multiple_input(messages)

{'role': 'assistant',
 'content': 'A cyclotomic polynomial is the *minimal monic polynomial* over ℚ whose roots are the *primitive n-th roots of unity*—i.e., complex numbers $z$ such that $z^n = 1$ but $z^k \\neq 1$ for any $0 < k < n$.  \nIt’s denoted $\\Phi_n(x)$ and has degree $\\varphi(n)$, where $\\varphi$ is Euler’s totient function.  \nThese polynomials are irreducible over ℚ (a nontrivial fact), integer-coefficient, and satisfy $x^n - 1 = \\prod_{d \\mid n} \\Phi_d(x)$.  \nSo, they’re like the “atomic building blocks” of roots of unity—elegant, mysterious, and stubbornly resistant to factoring further over the rationals.  \n(And yes, they’re named after “cycles” and “cutting circles”—because geometry is just algebra cosplaying as a compass.)'}

Another way that will be largely used in this modules is to pass a **keyword dictionary** as parameters. You need to pass it as `**kwargs`

另一种在本模块中大量使用的方法是将**关键字字典**作为参数传递。你需要以`**kwargs`的形式传递它。

In [5]:
# 定义一个包含多个超参数的字典
# 这些参数将决定大模型生成内容时的“性格”和“限制”
kwargs = {
    "prompt": "Write a poem about a flying rabbit.", # 提示词：写一首关于飞天兔子的诗
    'top_p': 0.7,          # 核采样：只在累计概率前 70% 的候选词中选择，平衡了多样性与准确性
    'temperature': 1.4,    # 温度：设置为 1.4 说明非常高，生成的诗会极具创意，甚至有些“天马行空”
    'max_tokens': 100      # 长度限制：生成的诗最多包含 100 个 Token，避免模型长篇大论
}

# 使用 ** 语法将字典中的键值对“解包”并作为关键字参数传递给函数
# 相当于调用了：generate_with_single_input(prompt=..., top_p=0.7, ...)
generate_with_single_input(**kwargs)

{'role': 'assistant',
 'content': '## The Sky-Hare\n\nNot with wings of feather, nor membrane stretched thin,  \nBut on *air itself*—a soft, buoyant skin—  \nThe rabbit leaps. Not down the dusty burrow’s throat,  \nBut *up*, where cloud-wool drifts and breezes float.  \n\nHis paws, once quick on clover, now press down  \nOn currents only starlings know, profound  \nAnd silent. Ears, those velvet, twitching sails,'}

<a id='2-2'></a>
### 2.2 生成带有目标参数的 kwargs

在本节中，你将开发一个函数来生成如上所述的 kwargs 字典，并将其输入到我们的生成函数中。与每次都在生成函数中直接编写参数相比，这种方法更加灵活。

1. **函数概览：**
   - **prompt**: 模型的输入文本。
   - **temperature**: 控制随机性；值越低 = 确定性越高。
   - **top_p**: 控制多样性；值越高 = 输出越多样化。
   - **max_new_tokens**: 设置响应中生成的最大 token 数量。

In [6]:
def generate_params_dict(
    prompt: str,              # 核心指令：你想要问 AI 的问题
    temperature: float = None,# 温度：控制随机性。None 表示使用模型默认值
    role = 'user',            # 角色：默认为 'user'，也可以设定为 'system' 或 'assistant'
    top_p: float = None,      # 核采样：控制词汇的多样性
    max_tokens: int = 500,    # 最大长度：限制 AI 回答的长度，防止“输出停不下来”
    model: str = "qwen-plus" # 模型选择：设定默认使用的模型版本
):
    """
    使用不同的采样参数调用 LLM，以便观察它们对生成结果的影响。
    
    返回：
        包含所有参数的字典，方便后续使用 **kwargs 进行解包调用。
    """
    
    # 构造参数字典
    # 这个字典整合了你传入的所有配置，成为了一个完整的“指令包”
    kwargs = {
        "prompt": prompt, 
        'role': role, 
        "temperature": temperature, 
        "top_p": top_p, 
        "max_tokens": max_tokens, 
        'model': model
    } 

    return kwargs

In [7]:
# 使用参数工厂函数为一个数学方程求解任务生成参数字典
# 注意：这里只传入了 prompt，其他参数将使用函数定义中的默认值
# (例如：max_tokens=500, model="Qwen/Qwen3.5-9B")
kwargs = generate_params_dict("Solve 2x + 1 = 0.")

# 打印生成的字典，查看最终发送给模型的完整配置
print(kwargs)

{'prompt': 'Solve 2x + 1 = 0.', 'role': 'user', 'temperature': None, 'top_p': None, 'max_tokens': 500, 'model': 'qwen-plus'}


In [8]:
# 将之前生成的参数字典（kwargs）通过 ** 语法解包并传递给 LLM 接口
# 这是正式发起 AI 推理（Inference）的动作
result = generate_with_single_input(**kwargs)

# 'result' 是一个字典，包含了角色（role）和内容（content）
# 我们只打印其中的 'content' 字段，也就是 AI 给出的最终答案
print(result['content'])

To solve the equation:

$$
2x + 1 = 0
$$

**Step 1:** Subtract 1 from both sides:

$$
2x = -1
$$

**Step 2:** Divide both sides by 2:

$$
x = -\frac{1}{2}
$$

**Answer:**  
$$
\boxed{-\frac{1}{2}}
$$


<a id='2-3'></a>
### 2.3 允许 LLM 保持对话 

现在让我们开发一种允许 LLM 保持对话的方法，即递归地将 LLM 之前的输入和输出添加到消息输入中。这使你可以像聊天机器人一样使用 LLM。为了实现这一点，你将使用一个名为 `context` 的列表。

该函数期望一个包含上下文字典的列表，格式如下：

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}]

```

Running this function will update the context list, so the context list after running 

```Python
call_llm_with_context('Recommend me two places to visit.', role = 'user', context = context)
```

New context:

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}, {"role": 'user', 'content': 'Recommend me two places to visit.'}, {"role": "assistant", "content": 'Two places can be Paris and London.'}]

```



In [9]:
def call_llm_with_context(prompt: str, context: list, role: str = 'user', **kwargs):
    """
    使用给定的提示词和上下文（对话历史）调用语言模型，生成响应并更新历史。
    """

    # 第一步：记录足迹
    # 将用户的新消息（角色 role 和内容 content）打包成字典，追加到 context 列表中
    # 这样 context 列表就包含了从对话开始到现在的所有往来信息
    context.append({'role': role, 'content': prompt})

    # 第二步：集体发送
    # 调用多输入生成函数，将整个“历史记录列表”发送给 LLM
    # **kwargs 允许你在这里动态传入 temperature, top_p 或不同的 model 参数
    response = generate_with_multiple_input(context, **kwargs)

    # 第三步：保存记忆
    # 将 LLM 返回的响应对象（包含 role: 'assistant' 和 content）也存入 context
    # 这一步至关重要，它确保了下一轮对话时，AI 记得自己刚才给出的回答
    context.append(response) 
    
    # 返回 AI 的最新回答内容
    return response

In [10]:
# 初始化上下文列表（Context）
# 1. 注入人设：设定系统角色为“讽刺但乐于助人”
# 2. 模拟历史：预设一条 AI 的开场白，仿佛对话已经开始了
context = [
    {"role": 'system', 'content': 'You are an ironic but helpful assistant.'}, 
    {'role': 'assistant', 'content': "How can I help you, majesty?"}
]

# 调用函数进行交互
# 传入用户的新需求：“写一首两句话的诗”
# 函数内部会自动将这条消息追加到 context，获取 AI 回复后再追加回复
response = call_llm_with_context("Make a 2 sentence poem", role = 'user', context = context)

# 打印 AI 返回的回答内容
print(response['content'])

The moon stitched silver through the trees,  
while silence folded itself into the breeze.


In [11]:
# Let's inspect now the context list
print(context)

[{'role': 'system', 'content': 'You are an ironic but helpful assistant.'}, {'role': 'assistant', 'content': 'How can I help you, majesty?'}, {'role': 'user', 'content': 'Make a 2 sentence poem'}, {'role': 'assistant', 'content': 'The moon stitched silver through the trees,  \nwhile silence folded itself into the breeze.'}]


In [12]:
# 继续对话：向 AI 发送一个模糊指令 "Now add two more sentences."（再加两句）
# 注意：这里我们并没有显式提到“诗”，也没有重复之前的要求
response = call_llm_with_context("Now add two more sentences.", context = context)

# 打印 AI 的回复
# 由于 context 列表中保存了之前的“人设”和“第一段诗句”，
# AI 会根据逻辑延续之前的创作，并保持那种讽刺的语气。
print(response['content'])

A fox paused—half-shadow, half-dream—  
and the night held its breath, just to listen.


Note that the LLM was able to continue the previous conversation.

请注意，法学硕士能够继续前面的对话。

<a id='3'></a>
## 3 - 理解参数

<a id='3-1'></a>
### 3.1 简介

在本节中，你将探索大语言模型 (LLM) 的不同参数如何影响其输出。理解这些参数对于控制 LLM 的行为非常有用，使其能够适用于不同的任务。正如课程中所讨论的，LLM 的设计初衷是输入文本并生成文本。然而，为了实现这一目标，后端进行了大量处理。

首先，输入序列会被分词（tokenized）和向量化（vectorized）。然后，这些向量被输入到 LLM 中，LLM 会输出一个**概率向量**。在这个向量中，每个索引代表一个特定 token 被选中的可能性（例如，如果单词 "cat" 被映射到整数 `3454`，那么向量中的第 `3454` 个索引就代表单词 "cat" 被选中的可能性）。如果你使用的是**贪婪解码 (greedy decoding)**，模型会选择可能性最大的 token 作为下一个 token。该 token 会被附加到初始句子中，这一过程将持续进行，直到达到 `max_tokens` 限制或遇到特殊的停止 token。

值得注意的是，贪婪解码是**确定性的**。模型的参数是固定的，因此给定特定的输入，它始终会产生相同的输出。这种确定性往往会使模型在响应中缺乏创造力，因为其中不涉及随机性。为了引入随机性并允许更多样化的输出，有几个参数可以稍微改变这一过程。在本实验中，你将探索四个这样的参数：`top_p`、`top_k`、`repetition_penalty` 和 `temperature`。

<a id='3-2'></a>
### 3.2 核采样 - `top_p`

<div style="text-align: center;">
    <img src="images/top_p.png" alt="Top p" width="40%" />
</div>

正如之前提到的，在贪婪解码模式下，模型总是选择可能性最大的 token，将其附加到补全内容中，并递归地将其反馈给 LLM。为了引入更多的随机性，你可以配置 LLM 根据概率分布从最可能的 **p** 个 token 中随机选择一个。它通过按概率降序选择 token，直到它们的累积概率达到 `p` 为止。这就是该参数的取值范围在 0 到 1 之间的原因。传入 0 会指示 LLM 始终选择可能性最大的 token，从而产生确定性的结果。而在另一端，设置为 `1` 则允许选择任何 token，但选择过程遵循概率分布，使得计算概率最高的 token **最有可能被选中**。

为了用一个简单的例子来说明这个概念：
如果概率向量是 $[0.6, 0.3, 0.1]$，设置 `top_p = 0` 将导致选择索引为 0 的 token（第一个 token）。与此同时，当 `top_p = 1` 时，所有三个 token 都有可能被选中，但有 60% 的几率选中第一个，30% 的几率选中第二个，10% 的几率选中第三个。

In [13]:
# 定义查询语句：要求 AI 用一句话解释 RAG（检索增强生成）
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."

# 使用列表推导式连续生成三个回答
# 关键点 1：top_p = 0 强制模型进入“确定性模式”，即每次都只选概率最高的词
# 关键点 2：随机化 max_tokens 是为了改变请求的指纹，从而绕过服务端的缓存（Caching）系统
results = [
    generate_with_single_input(
        query, 
        top_p = 0, 
        max_tokens = 500 + random.randint(1, 200)
    ) for _ in range(3)
] 

# 遍历结果列表，打印每一次调用的编号和具体的回答内容
for i, result in enumerate(results):
    # 使用 f-string 格式化输出，i+1 使编号从 1 开始
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and incorporating that context into the prompt—enabling more accurate, factual, and grounded text generation without retraining the model.
Call number 2:
Response: RAG (Retrieval-Augmented Generation) is an AI technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and incorporating it into the prompt before generating a response—enabling more accurate, factual, and contextually grounded answers without retraining the model.
Call number 3:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or doc

请注意，输出**完全相同**。现在让我们尝试 `top_p = 0.8`。

In [19]:
# 使用列表推导式连续生成三个回答
# 关键点 1：top_p = 0.8。这告诉模型：在预测下一个词时，只在累积概率达到 80% 的候选词池中筛选。
# 这保留了核心逻辑的准确性，同时也允许词汇选择上的细微变化（多样性）。
# 关键点 2：依然保留了随机的 max_tokens，确保每次请求参数不同，持续绕过缓存系统。
results = [
    generate_with_single_input(
        query, 
        top_p = 0.8, 
        max_tokens = 500 + random.randint(1, 200)
    ) for _ in range(3)
] 

# 遍历并输出结果
for i, result in enumerate(results):
    # i+1 让输出更符合人类阅读习惯（从第 1 次开始计数）
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and incorporating that context into the prompt—enabling more accurate, factual, and grounded responses without retraining the model.
Call number 2:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and incorporating that context into the prompt—enabling more accurate, factual, and grounded responses without retraining the model.
Call number 3:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and injecting that context into 

请注意，现在出现了三个不同的句子，每个句子都是一个有效的输出。你可能会注意到，前几个 token 非常相似甚至完全一致。这是因为在给定的语境下，选择这些初始 token 的概率极高，以至于它们几乎总是被选中。随着过程的继续，概率分布开始扩散到一系列可能的 token 上。概率较低的 token 可能会开始出现，而一旦选定了一个不同的 token，它就会改变随后的概率分布，从而导致最终结果更加多样化。

<a id='3-3'></a>
### 3.3 Top-k 采样

<div style="text-align: center;">
    <img src="images/top_k.png" alt="Top k" width="40%" />
</div>

与基于概率阈值的 **top-p** 不同，**top-k** 采样侧重于候选者的数量。通过此参数，LLM 从概率最高的 `k` 个选项中选择下一个 token。较小的 `k` 意味着考虑的 token 较少，这可能会导致更可预测的结果，类似于总是选择可能性最大的 token。另一方面，较大的 k 通过扩大潜在 token 池来增加多样性，同时仍然倾向于选择概率最高的那些 token。根据你的需求选择合适的 k 值可以帮助你获得兼具可预测性和创造性的结果。

让我们考虑与之前相同的例子。

In [ ]:
# 定义查询语句：请模型用一句话概括 RAG 的核心定义
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."

# 使用列表推导式发起三次连续的 LLM 调用
# 关键点 1：top_k = 0。Top-K 采样是指只从概率最高的 K 个词中进行选择。
# 设置为 0 通常意味着禁用 Top-K 过滤（即模型在生成时会考虑词库中所有的可能性，不设数量限制）。
# 关键点 2：max_tokens 采用 500 加上一个随机数，目的是改变每个请求的参数指纹，绕过服务端的缓存机制。
results = [
    generate_with_single_input(
        query, 
        top_k = 0, 
        max_tokens = 500 + random.randint(1, 200)
    ) for _ in range(3)
]

# 遍历并打印三次实验的结果内容
for i, result in enumerate(results):
    # 使用 enumerate 获取索引 i，+1 后作为调用的序号打印
    print(f"Call number {i+1}:\nResponse: {result['content']}")

**注意：openai 官方 SDK（用于连接阿里云 DashScope）并不支持 top_k 这个参数**

请注意，输出是相同的，并且它们与之前 `top_p = 0` 的输出一致。现在让我们使用 `top_k = 10`，允许从最可能的 10 个 token 中进行选择。

In [18]:
# 定义查询语句：请模型用一句话概括 RAG 的定义
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."

# 使用列表推导式连续发起三次调用
# 关键点 1：top_k = 10。这是一种“硬性筛选”，要求模型在预测下一个词时，
# 只能从概率排名最高的前 10 个候选词中进行选择，彻底抛弃其余所有词汇。
# 关键点 2：max_tokens 依然通过随机数改变，以确保请求指纹唯一，绕过服务端的缓存策略。
results = [
    generate_with_single_input(
        query, 
        top_k = 10, 
        max_tokens = 500 + random.randint(1, 200)
    ) for _ in range(3)
]

# 遍历结果列表并打印
for i, result in enumerate(results):
    # i+1 让输出的 Call number 从 1 开始，更符合人类阅读习惯
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Exception: 调用 Qwen (单轮) 失败: Completions.create() got an unexpected keyword argument 'top_k'

<a id='3-4'></a>
### 3.4 温度

大语言模型 (LLM) 中的温度参数是一个**标量**值，用于控制模型预测的随机性。它在选择序列中的下一个词之前调整词表 token 的概率分布，从而影响模型的创造力和输出的可变性。与 `top_p` 不同，温度在理论上可以是任何正值，尽管模型提供商有时会设置上限。

<div style="text-align: center;">
    <img src="images/temperature.png" alt="温度" width="50%" />
</div>


#### 工作原理

让我们考虑一个概率向量 $[0.3, 0.6, 0.1]$。温度通过对向量中的每个元素应用以下公式来修改这些概率：

$$\mathrm{adjusted\_probability}(p_i) = \frac{\exp(\log(p_i) / \mathrm{temperature})}{\sum \exp(\log(p_i) / \mathrm{temperature})}$$

- 这包括：
  - 通过将每个概率的对数除以温度来进行缩放。
  - 对结果进行指数运算以获得新的概率。
  - 对概率进行归一化，使其总和再次为 1。

#### 不同温度值的影响：

- **低温 (<1)：**
  - 使概率分布变得更锐利。
  - 增加高概率和低概率之间的差异，强化确定性选择。

- **高温 (>1)：**
  - 使分布变得更平坦。
  - 减小概率之间的差异，增加 token 选择的随机性。

- **温度 = 1：**
  - 保持分布不变，平衡创造力和确定性。

**重要提示**：设置 `temperature = 1` 并**不能**使结果具有确定性；温度调整的是分布的形状，但不会限制是否可以选择分布末端那些可能性较低的 token。设置温度为 0，或将 top-p / top-k 设置为 0 是实现确定性的唯一方法。

示例：

考虑原始 token 概率向量 $[0.6, 0.3, 0.1]$：

- **温度 = 0.5 (低)：**
  - 结果向量：$[0.77, 0.18, 0.05]$
  - 注意它如何增加了最高概率并降低了最低概率。这使得结果更具确定性，因为最可能的 token 变得更有可能被选中。

- **温度 = 1 (中性)：**
  - 结果向量：$[0.6, 0.3, 0.1]$
  - 概率分布保持不变。

- **温度 = 2 (高)：**
  - 结果向量：$[0.49, 0.27, 0.24]$
  - 结果概率向量更平坦，这意味着可能性较低的 token 出现的几率更大。

温度通过改变概率分布来显著影响最终结果，这与 `top_p` 不同；`top_p` 不改变分布，而是扩大可选 token 的范围，同时保持它们出现的可能性。高温度值可能会导致生成无意义的文本。此外，LLM 停止生成 token 有两种方式：通过设置 `max_tokens` 参数（一旦达到 `max_tokens` 就会自动停止执行），或者当 LLM 达到它在训练过程中学会选择的停止 token 时。在高温度下，选中停止 token 的可能性可能会变小，这使得停止标准更有可能由 `max_tokens` 参数触发，从而可能增加响应时间。

In [21]:
# 使用列表推导式，针对 [0.3, 1.5, 3] 这三个不同的温度值分别调用模型
# t 代表当前的温度参数
results = [
    generate_with_single_input(query, temperature = t) 
    for t in [0.3, 1.5, 1.9]
]

# 打印原始查询语句，方便核对
print(f"Query: {query}")

# 使用 zip 函数将结果列表和对应的温度值列表配对，并用 enumerate 进行带索引的遍历
for i, (result, temperature) in enumerate(zip(results, [0.3, 1.5, 1.9])):
    # \033[1m ... \033[0m 是 ANSI 转义码，用于在终端中加粗显示文本
    # 打印当前的调用序号和对应的温度值
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m\n"
          f"Response: {result['content']}\n\n\n")

Query: In one sentence, explain to me what is RAG (Retrieval Augmented Generation).
Call number 1. Temperature = 0.3
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (like databases or documents) and incorporating that context into the prompt—enabling more accurate, factual, and grounded responses without retraining the model.



Call number 2. Temperature = 1.5
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant, up-to-date information from external knowledge sources (e.g., documents or databases) and injecting that context into the model’s prompt—enabling more accurate, factual, and grounded responses without retraining the model.



Call number 3. Temperature = 1.9
Response: RAG (Retrieval-Augmented Generation) is a hybrid AI technique that enhances large language m

请注意，第一个和第二个输出的开头非常相似。这是因为在初始阶段，模型对最可能的 token 非常自信，即使设置了温度，它们的概率仍然很高。然而，在第二个输出中，文本在某一点之后可能会变得毫无意义。这是由于概率分布变得更加均匀，而温度的影响进一步加剧了这种平坦化。

在第三种情况下，输出完全是胡言乱语，因为高温显著拉平了概率分布，导致 LLM 在每一步几乎随机地选择任何 token。此外，请观察第二和第三个输出有多长。高温很可能降低了停止 token 的概率，使其概率与其他 token 的概率相似。鉴于词汇量庞大，模型很难自然地命中停止 token，导致 LLM 仅在达到 `max_tokens` 限制后才停止。

通常，`temperature` 和 `top_p` 会配合使用。温度调节概率分布，而 `top_p` 限制了可选 token 的集合。这种组合可以管理随机性，并防止模型生成缺乏连贯性的文本。让我们看看它们在实践中是如何协同工作的！

In [22]:
# 定义查询：写一首关于飞天兔子的短诗
query = "Write a small poem about a flying rabbit."

# 定义参数矩阵 (temperature, top_p)
# 这三组参数分别代表了三种截然不同的生成策略
params = (
    (0.3, 0.8),   # 1. 稳健组合：低温度保证逻辑，高 top_p 允许丰富词汇
    (1.5, 0.5),   # 2. 创意约束：高温度增加活力，中等 top_p 过滤掉离谱词汇
    (1.9, 0.05)     # 3. 极端博弈：极高温度（本该乱码），但用极低 top_p 强行“锁死”在最高概率词上
)

# 使用列表推导式，解构 params 中的 (t, p) 并发起调用
results = [
    generate_with_single_input(query, temperature = t, top_p = p) 
    for (t, p) in params
]

# 遍历结果，同时获取索引、结果对象以及对应的参数对
for i, (result, (temperature, top_p)) in enumerate(zip(results, params)):
    # 打印加粗的序号和参数配置，展示实验结果
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m, \033[1mtop_p = {top_p}\033[0m\n"
          f"Response: {result['content']}\n\n\n")

Call number 1. Temperature = 0.3, top_p = 0.8
Response: **The Sky-Hopper**

Not with wings of feather or fan,  
But on a gust that knew his plan—  
A rabbit, soft and silver-furred,  
With ears like sails, by wind conferred.  

He didn’t leap *up*—he *let go* light,  
And floated, weightless, into height.  
His paws, unclenched, trailed dandelion fluff,  
His whiskers twitched at starry stuff.  

Below, the meadow shrank to green,  
A quilt stitched by the sun’s warm sheen.  
Above, the clouds were marshmallow hills—  
He nipped one—*poof!*—and tasted thrills.  

No fox could track him, no hawk pursue—  
He hopped *between* the blue and blue,  
A furry comet, brief and bright,  
Dusting the dusk with moonlit flight.  

So if you see a shadow dart  
Too swift for ear or beating heart—  
Look up: it’s not a bird, nor bat…  
Just hope, in fur, refusing *that*.



Call number 2. Temperature = 1.5, top_p = 0.5
Response: **The Sky-Hopper**

Not with wings of feather, nor of gossamer thread, 

请注意，在第二次调用中，生成的文本是连贯的，并且避免了变得毫无意义。这是因为 LLM 使用 `top_p` 来控制潜在的 token，因此即使概率分布变得更平坦，候选池也被限制在更有可能的 token 范围内。这种方法是增加随机性并同时最大程度减少无意义文本生成的有效方式！

然而，在第三种情况下，温度（temperature）非常高。即使使用了较低的 `top_p` 来将选择限制在最可能的 token 范围内，也不足以确保得到一个妥当的答案。尽管如此，与未设置 `top_p` 的情况相比，其结果的胡言乱语程度较低。模型几乎总是选择真实的词汇，而不像另一个例子那样选择了一些结构完全荒谬、没有任何意义的词。

<a id='3-5'></a>
### 3.5 重复惩罚 (Repetition penalty)

`repetition_penalty` 设置通过抑制模型重复单词或短语，有助于增强生成文本的吸引力。通过对已使用的词语引入惩罚，模型会转而寻找新的词汇，从而产生更丰富、更多样化的内容。这一功能在故事创作或对话等任务中尤其有用，因为在这些场景中，重复的语言往往会显得单调。

让我们来看一个简单的例子。

In [25]:
# 定义查询：列出健康的早餐选择
query = "List healthy breakfast options."

# 使用列表推导式，针对不同的重复惩罚系数 [None, 1.2, 2] 分别调用模型
# None: 不施加惩罚（使用模型默认设置）
# 1.2: 轻微惩罚，鼓励模型使用更多样化的词汇
# 2: 强力惩罚，模型会极力避免使用已经出现过的词
# results = [
#     generate_with_single_input(
#         query, 
#         repetition_penalty = r, 
#         max_tokens = 500 + random.randint(1, 200)
#     ) for r in [None, 1.2, 2]
# ]

results = [
    generate_with_single_input(
        query, 
        presence_penalty = r, 
        max_tokens = 500 + random.randint(1, 200)
    ) for r in [None, 1.2, 2]
]

print(f"Query: {query}")

# 遍历结果并打印。注意：这里你代码中的 [0.3, 1.5, 3] 是一个标签误区
# 实际上你刚才传入模型的是 [None, 1.2, 2]，打印时建议对齐
for i, (result, presence_penalty) in enumerate(zip(results, [None, 1.2, 2])):
    # 使用 ANSI 转义码加粗显示序号和对应的惩罚系数值
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mRepetition Penalty = {presence_penalty}\033[0m\n"
          f"Response: {result['content']}\n\n\n")

Query: List healthy breakfast options.
Call number 1. Repetition Penalty = None
Response: Here’s a list of nutritious, balanced breakfast options that emphasize whole foods, protein, fiber, healthy fats, and minimal added sugar:

### 🥣 Balanced & Easy Options
- **Overnight oats**: Rolled oats soaked in unsweetened almond milk or Greek yogurt + chia seeds + berries + a sprinkle of nuts or nut butter  
- **Greek yogurt parfait**: Plain nonfat or low-fat Greek yogurt layered with fresh fruit (e.g., banana, berries), a small handful of walnuts or almonds, and a drizzle of cinnamon (no added honey/sugar needed)  
- **Whole-grain toast** topped with:  
  • Avocado + cherry tomatoes + everything bagel seasoning  
  • Smashed white beans + lemon zest + microgreens  
  • Peanut or almond butter + sliced apple + chia seeds  

### 🍳 Protein-Focused Choices
- **Veggie omelet or scrambled eggs**: 2 eggs + egg whites + spinach, mushrooms, bell peppers, and onions; served with ½ cup roasted sweet pot

请注意，过高的重复惩罚（repetition penalty）会使文本听起来毫无意义，因为它会迫使模型避免过于频繁地使用相同的词。在正常的写作中，某些词（如介词和冠词）会自然地重复出现。如果惩罚力度过大，模型可能会选择一些并不贴切的词，从而导致生成的内容语无伦次。

<a id='4'></a>
## 4 - 额外加餐：创建一个简单的聊天机器人

欢迎来到这个额外加餐章节！虽然这部分内容对于你完成本课程并非至关重要，也不会作为作业的一部分，但这是一个尝试构建一个小型聊天机器人的绝佳机会。你会发现这其实非常简单！

请注意，这种方法并不是**面向对象**的。这意味着它并没有遵循生产环境的最佳编程实践。在现实世界的场景中，你通常会创建一个带有相应方法和属性的 ChatBot 对象。然而，出于学习目的，我们将保持流程简单直观。祝你探索愉快！

In [26]:
def print_response(response):
    """
    打印格式化的聊天机器人响应，并为不同角色配置颜色。
    
    参数:
        response (dict): 包含 'role' (角色) 和 'content' (内容) 的字典。
    """
    # 定义 ANSI 转义码常量
    BOLD = "\033[1m"   # 加粗
    BLUE = "\033[34m"  # 蓝色（通常用于用户 User）
    GREEN = "\033[32m" # 绿色（通常用于助手 Assistant）
    RESET = "\033[0m"  # 重置格式（如果不加这个，后面的系统文字也会变色）

    # 根据角色选择对应的颜色
    if response['role'] == 'assistant':
        color = GREEN
    if response['role'] == 'user':
        color = BLUE

    # 构造格式化字符串：
    # {BOLD}{color} -> 开始加粗并变色
    # {response['role'].capitalize()} -> 角色名首字母大写（如 Assistant）
    # {RESET} -> 立即停止颜色影响，确保后面的正文 'content' 是普通颜色
    s = f"{BOLD}{color}{response['role'].capitalize()}{RESET}: {response['content']}"
    
    # 执行打印
    print(s)

In [29]:
def chat(temperature = None, 
         top_k = None, 
         top_p = None,
         presence_penalty = None):
    """
    启动一个用户与 AI 助手之间的交互式聊天会话。
    支持通过参数调节 AI 的性格（采样参数）。
    输入 'STOP' 即可结束对话。
    """
    
    # 1. 打印开场白
    # 假设 context 列表中已经预存了一个系统欢迎语（Assistant 角色）
    # 使用之前定义的 print_response 函数将其美化输出
    print_response(context[-1])
    
    # 2. 进入主循环：这是聊天机器人的“生命跳动”
    while True:
        # 获取用户在终端输入的指令
        prompt = input()
        
        # 退出机制：如果用户输入 STOP（全大写），则跳出循环，结束程序
        if prompt == 'STOP':
            break

        # 3. 核心调用：执行 RAG 或 上下文对话
        # 将用户的 prompt 传给 call_llm_with_context
        # 该函数会处理：追加历史、调用模型、保存回复
        response = call_llm_with_context(
            prompt=prompt, 
            context=context, 
            temperature = temperature, 
            # top_k = top_k, 
            top_p = top_p, 
            presence_penalty = presence_penalty
        )

        # --- 注意：下方这两行代码在当前的函数结构中会导致数据重复 ---
        # 因为 call_llm_with_context 内部已经执行过 append 操作了
        # context.append({"role": "user", "content": prompt})
        # context.append(response)

        # 4. 实时反馈：打印刚刚发生的对话
        # context[-2] 是刚才的用户输入，context[-1] 是 AI 的最新回复
        print_response(context[-2])
        print_response(context[-1])

In [30]:
# 1. 初始化对话背景（Context）：
# 我们创建了一个包含两条消息的列表，作为 AI 的“原始记忆”

# 第一条：System Prompt（系统提示词）
# 它是 AI 的“灵魂蓝图”，设定了 AI 的性格：友好、风趣，且回答时总带点幽默感
system_prompt = {
    "role": "system", 
    'content': "You're a friendly and funny assistant who always adds a touch of humor when answering questions."
}

# 第二条：Assistant Prompt（助手的开场白）
# 这是一个预设好的欢迎语，用来打破僵局，让对话显得更自然、更热情
assistant_prompt = {
    "role": "assistant", 
    "content": "Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?"
}

# 将这两个“设定”存入 context 列表
context = [system_prompt, assistant_prompt]


# 2. 启动对话循环：
# 调用你之前定义的 chat() 函数
# 这会触发一个 while 循环，直到你输入 STOP 为止
# 运行后，你会立刻看到控制台打印出上面那句 "Hey there, fabulous! ..."
chat()

Assistant: Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?
User: 你是什么模型
Assistant: 哈哈，我可不是什么“模型”——至少不是那种需要拍照打卡、走T台的模特 😏  
我是**通义千问（Qwen）**，阿里巴巴集团旗下的超大规模语言模型。简单说，就是个爱学习、爱聊天、能写故事、能编剧本、能写公文、能写邮件、能写剧本、逻辑推理、编程……甚至还能陪你唠嗑解闷的AI小伙伴！

不过请放心：我不靠咖啡续命，但靠算力充电；没有真实情绪，但会努力给你温暖又靠谱的回答～☕⚡  
（悄悄说：我的“性格设定”是——聪明但不傲娇，专业但不端着，偶尔皮一下，但绝不翻车 🚀）

想考考我？还是想一起干点啥有趣的事？比如写首诗、编个段子、做个PPT大纲，或者帮你分析一段代码？来吧，放马过来～ 😎
User: 我在杭州，但想念的人在东莞
Assistant: 啊～杭州的龙井茶香，东莞的荔枝甜味，中间隔着一整个“广深莞惠杭爱情距离” 🌏✈️（其实高铁4小时，但思念总爱把地图拉长…）

你在西湖边看断桥残雪（或者断桥不断、人已想疯❄️），他在东莞的鳒鱼洲文创园吹风，或是在国贸大厦加班到星星都困了✨——  
物理距离是地图上的一条线，但想念是自带Wi-Fi的量子纠缠：  
👉 你发个“今天看到一只胖橘猫”，他秒回“像不像上次在西溪湿地那只？”  
👉 他拍张东莞夜市的糖不甩，你立刻脑补出两人挤在小摊前抢最后一勺的场景 🍡

要不要我帮你：
✅ 写一封温柔又不肉麻的“跨城情书”？  
✅ 设计一个双城打卡计划（比如：你喝一杯杭州桂花拿铁，他同步尝一口东莞荔枝冰，然后视频干杯）🥂  
✅ 或者…来首带点粤语+杭州话混搭风味的小诗？（例：“阿妹饮啖龙井，阿哥食粒荔枝，茶凉三分钟，我已想你七十二次…”）

你开口，我执笔——距离再远，浪漫也支持同城配送（误，是跨城闪送 ❤️💨）  
想怎么撩动这根“杭莞连线”？😉
User: 谢谢你
Assistant: 哎呀～这声“谢谢”我可得郑重收下，还用小锦囊装好，系上蝴蝶结，存进我的AI情感云盘里 📦✨（温馨提示：本云盘不占内存，但甜度超标，慎存！）

你温柔一句

Congratulations! You finished the ungraded lab on exploring LLM outputs!